<a href="https://colab.research.google.com/github/dee0742/ML-FlyRank-Task/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dee0742/ML-FlyRank-Task/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is **ranking**.

The decision is: **which content items should be reviewed first?**

The output will be a priority score for each content item. Items with higher scores would be placed higher in the review queue.

This is a ranking task because the main goal is to decide which items come first, rather than only predicting a yes/no outcome.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target should represent an observed outcome that can help measure whether a content item needs attention.

For this starter dataset, I will use an observed performance outcome as the proxy for content priority rather than using `is_declining_label` as the target.

`is_declining_label` is derived from `trend_direction` and `trend_pct`, so using it as the target would mean learning a defined rule rather than predicting an independently observed outcome.

The target should therefore come from an observed performance measurement in the available data.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use **Precision@K** as the success metric.

Precision@K measures how many of the top K content items are actually high-priority items according to the observed outcome.

This matches the decision because the team will act on the first few items in the review queue.

I will calculate the metric against the observed outcome rather than against a label created from the same input signals.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one **pseudonymized content item**.

The starter dataset contains 30,000 content items and 44 columns. Each content item has trailing-90-day performance and content-related measurements.

This unit matches the decision because the review team needs to prioritize individual content items.

In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/dee0742/ML-FlyRank-Task/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
print("Columns:")
print(df.columns.tolist())

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [11]:
df[["content_id", "client_id", "content_type"]].head(10)

,content_id,client_id,content_type
0,content_304f48230142,client_f369cb89fc,keyword article
1,content_a1fb4e703a9e,client_4e07408562,keyword article
2,content_9aa793d4d895,client_7f2253d7e2,keyword article
3,content_331d6c4de07b,client_19581e27de,keyword article
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article
5,content_d4084a4bc775,client_f369cb89fc,keyword article
6,content_9a34b442b552,client_8722616204,keyword article
7,content_a63219c6e95a,client_19581e27de,keyword article
8,content_5e6c160719bc,client_6208ef0f77,keyword article
9,content_c27558df2b0c,client_19581e27de,keyword article


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could prioritize content using one metric and one manually chosen threshold.

However, content performance depends on multiple signals, and those signals can interact in ways that are difficult to describe with a small number of if-statements.

ML can combine several observed signals to produce a priority score and learn patterns from historical data.

The output supports a real decision: deciding which content items should be reviewed first.

The result will be treated as **decision-support**, not as proof that a highly ranked item definitely needs a refresh.

In [12]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [13]:
df.dtypes

,0
content_id,object
client_id,object
search_volume,float64
competition,float64
competition_level,object
cpc,float64
content_type,object
main_intent,object
word_count,float64
char_count,float64


In [14]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
content_id,30000,30000,content_6880eb215048,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
client_id,30000,32,client_19581e27de,7008,NaN,NaN,NaN,NaN,NaN,NaN,NaN
search_volume,27532.0,NaN,NaN,NaN,158.882391,1518.270825,0.0,0.0,10.0,20.0,74000.0
competition,27532.0,NaN,NaN,NaN,0.146954,0.285241,0.0,0.0,0.0,0.13,1.0
competition_level,27390,3,LOW,22896,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpc,27532.0,NaN,NaN,NaN,0.485342,2.10156,0.0,0.0,0.0,0.0,100.36
content_type,30000,3,keyword article,27207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
main_intent,27626,4,informational,17235,NaN,NaN,NaN,NaN,NaN,NaN,NaN
word_count,22301.0,NaN,NaN,NaN,3107.760325,1452.382598,8.0,2413.0,2877.0,3666.0,9546.0
char_count,22301.0,NaN,NaN,NaN,20665.277835,10115.344042,40.0,15644.0,19116.0,24011.0,111158.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.